# 리트리버 파인튜닝 전/후 성능 비교

Ko-miracl `default[train]` 관련도 라벨로 dense 리트리버를 **미세조정(fine-tuning)** 하고,
**파인튜닝 전 vs 후**를 동일 평가셋·동일 지표로 비교합니다. (LangChain 미사용)

```
base 임베딩 모델
   ├─▶ (A) 파인튜닝 전 평가  ──┐
   ├─▶ train qrels 로 파인튜닝   ├─▶ 전/후 비교표 (Recall/MRR/nDCG/Hit @k)
   └─▶ (B) 파인튜닝 후 평가  ──┘
```

## 도메인 교체 대비 설계
- 데이터셋·컬럼·스플릿 이름을 **상단 `DATA` config 한 곳**에 모아, 다른 도메인 데이터로 바꿀 때 여기만 수정하면 되게 했습니다.
- 학습 데이터 형식 가정: **queries(질문) + qrels(질문–문서 관련도, score>0=정답)**. 이 형식(BEIR 스타일)이면 그대로 재사용 가능합니다.

> 파인튜닝은 `sentence-transformers`의 **MultipleNegativesRankingLoss**(in-batch negative 대조학습)를 사용합니다.
> 기본 모델은 L4에서 빠르게 학습되는 경량 한국어 임베딩 `jhgan/ko-sroberta-multitask` 입니다.

In [2]:
!nvidia-smi

Tue Jul 21 17:09:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 0. 설정 (도메인 교체 시 여기만 수정)

In [3]:
!pip -q install -U datasets chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 117.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 132.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 129.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━

In [4]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ===== 데이터셋 스키마 (다른 도메인으로 교체 시 이 dict만 수정) =====
DATA = {
    "dataset": "taeminlee/Ko-miracl",
    # corpus 서브셋
    "corpus_config": "corpus", "c_id": "_id", "c_title": "title", "c_text": "text",
    # queries 서브셋
    "queries_config": "queries", "q_id": "_id", "q_text": "text",
    # qrels(관련도) 서브셋
    "qrels_config": "default", "train_split": "train", "dev_split": "dev",
    "qr_qid": "query-id", "qr_cid": "corpus-id", "qr_score": "score",
}

# ===== 파인튜닝할 임베딩 모델 =====
BASE_MODEL     = "jhgan/ko-sroberta-multitask"  # 경량 한국어. e5류로 바꾸면 아래 프리픽스 설정
QUERY_PREFIX   = ""     # e5 계열이면 "query: "
PASSAGE_PREFIX = ""     # e5 계열이면 "passage: "

# ===== 하이퍼파라미터 =====
TOP_K   = 10
K_LIST  = [1, 5, 10]
N_TRAIN_QUERIES = 300      # 파인튜닝에 쓸 train 쿼리 수 (전체 train은 12,767)
N_EVAL_QUERIES  = 100      # 평가 dev 쿼리 수
NEG_POOL_SIZE   = 3000     # 평가용 코퍼스 네거티브 풀 (전체 149만 → 부분집합)
EPOCHS     = 1
FT_BATCH   = 32            # 파인튜닝 배치 (in-batch negative 수와 직결)
MAX_SEQ_LEN = 256
IDX_BATCH  = 128

Device: cuda


## 1. 데이터 로드

In [5]:
from datasets import load_dataset

q_dd = load_dataset(DATA["dataset"], DATA["queries_config"])
queries = {r[DATA["q_id"]]: r[DATA["q_text"]] for r in q_dd[list(q_dd.keys())[0]]}

qr_dd = load_dataset(DATA["dataset"], DATA["qrels_config"])
train_qrels = pd.DataFrame(qr_dd[DATA["train_split"]])
dev_qrels   = pd.DataFrame(qr_dd[DATA["dev_split"]])
print("queries:", len(queries), "| train qrels:", len(train_qrels), "| dev qrels:", len(dev_qrels))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

queries.jsonl:   0%|          | 0.00/219k [00:00<?, ?B/s]

Generating queries split:   0%|          | 0/2761 [00:00<?, ? examples/s]

train.jsonl:   0%|          | 0.00/641k [00:00<?, ?B/s]

dev.jsonl:   0%|          | 0.00/153k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12767 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/3057 [00:00<?, ? examples/s]

queries: 2761 | train qrels: 12767 | dev qrels: 3057


## 2. 학습/평가 부분집합 구성 + 코퍼스 수집

전체 코퍼스(149만) 대신 **학습 정답문서 + 평가 정답문서 + 네거티브 풀**만 스트리밍으로 모읍니다.
(이 셀이 네트워크 스캔이라 가장 오래 걸립니다. 빠르게 보려면 위 개수들을 줄이세요.)

In [6]:
QID, CID, SC = DATA["qr_qid"], DATA["qr_cid"], DATA["qr_score"]

def sample_pos_queries(qrels_df, n):
    pos = qrels_df[qrels_df[SC] > 0]
    qids = pos[QID].unique().tolist(); random.shuffle(qids)
    return qids[:n]

train_qids = sample_pos_queries(train_qrels, N_TRAIN_QUERIES)
eval_qids  = sample_pos_queries(dev_qrels,  N_EVAL_QUERIES)

# 정답셋: qid -> {cid: score}
def build_gold(qrels_df, qids):
    g = {}
    for _, r in qrels_df[qrels_df[QID].isin(qids)].iterrows():
        g.setdefault(r[QID], {})[r[CID]] = float(r[SC])
    return g

train_gold = build_gold(train_qrels, train_qids)   # 학습쌍 생성용
gold       = build_gold(dev_qrels,  eval_qids)      # 평가용

# 텍스트가 필요한 corpus id = 학습 정답 + 평가 정답
needed_cids = {c for g in [train_gold, gold] for rels in g.values()
                 for c, s in rels.items() if s > 0}
print("train 쿼리:", len(train_qids), "| eval 쿼리:", len(eval_qids),
      "| 필요한 정답 문서:", len(needed_cids))

train 쿼리: 300 | eval 쿼리: 100 | 필요한 정답 문서: 879


In [7]:
corpus_dd = load_dataset(DATA["dataset"], DATA["corpus_config"], streaming=True)
corpus_stream = corpus_dd[list(corpus_dd.keys())[0]]
CID_, TIT_, TXT_ = DATA["c_id"], DATA["c_title"], DATA["c_text"]

corpus_text, corpus_title = {}, {}
found, neg = set(), 0
for row in corpus_stream:
    cid = row[CID_]
    if cid in needed_cids and cid not in corpus_text:
        corpus_text[cid] = row[TXT_]; corpus_title[cid] = row[TIT_]; found.add(cid)
    elif neg < NEG_POOL_SIZE:
        corpus_text[cid] = row[TXT_]; corpus_title[cid] = row[TIT_]; neg += 1
    if len(found) >= len(needed_cids) and neg >= NEG_POOL_SIZE:
        break
print("수집 청크:", len(corpus_text), "| 정답 확보:", len(found), "/", len(needed_cids))

# 평가 인덱스에 들어갈 문서 집합 = 확보된 모든 청크
index_cids = list(corpus_text.keys())

수집 청크: 3879 | 정답 확보: 879 / 879


## 3. 임베딩 · ChromaDB · retriever · 평가 함수 (재사용)

In [8]:
from sentence_transformers import SentenceTransformer
import chromadb

def embed(model, texts, prefix, batch_size=IDX_BATCH, show_progress=False):
    texts = [prefix + t for t in texts]
    return model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=show_progress)

chroma_client = chromadb.Client()

def build_collection(name, model, index_cids):
    try: chroma_client.delete_collection(name)
    except Exception: pass
    col = chroma_client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    texts = [corpus_text[c] for c in index_cids]
    embs = embed(model, texts, PASSAGE_PREFIX, show_progress=True)
    for i in range(0, len(index_cids), IDX_BATCH):
        col.add(ids=index_cids[i:i+IDX_BATCH], embeddings=embs[i:i+IDX_BATCH].tolist(),
                documents=texts[i:i+IDX_BATCH])
    return col

def retrieve(col, model, query_text, k=TOP_K):
    q = embed(model, [query_text], QUERY_PREFIX, batch_size=1)
    res = col.query(query_embeddings=q.tolist(), n_results=k)
    return [{"corpus_id": cid, "score": 1 - dist}
            for cid, dist in zip(res["ids"][0], res["distances"][0])]  # score = 1 - distance

In [9]:
def dcg(rels):
    return sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(rels))

def evaluate(col, model, eval_qids, gold, k_list=K_LIST, top_k=TOP_K):
    names = ["Recall", "MRR", "nDCG", "Hit"]
    acc = {f"{m}@{k}": [] for k in k_list for m in names}
    for qid in eval_qids:
        ranked = [r["corpus_id"] for r in retrieve(col, model, queries[qid], k=top_k)]
        rel = {c for c, s in gold[qid].items() if s > 0}
        if not rel: continue
        for k in k_list:
            hits = [1 if c in rel else 0 for c in ranked[:k]]
            acc[f"Recall@{k}"].append(sum(hits) / len(rel))
            acc[f"Hit@{k}"].append(1.0 if any(hits) else 0.0)
            rr = next((1.0 / (i + 1) for i, h in enumerate(hits) if h), 0.0)
            acc[f"MRR@{k}"].append(rr)
            idcg = dcg([1] * min(len(rel), k))
            acc[f"nDCG@{k}"].append(dcg(hits) / idcg if idcg > 0 else 0.0)
    order = [f"{m}@{k}" for k in k_list for m in names]
    return {m: float(np.mean(acc[m])) if acc[m] else 0.0 for m in order}

## 4. (A) 파인튜닝 전 평가 — Baseline

In [10]:
model = SentenceTransformer(BASE_MODEL, device=DEVICE)
model.max_seq_length = MAX_SEQ_LEN

col_before = build_collection("retr-before", model, index_cids)
before = evaluate(col_before, model, eval_qids, gold)
print("파인튜닝 전:"); print(pd.Series(before).round(4))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

파인튜닝 전:
Recall@1     0.5052
MRR@1        0.8100
nDCG@1       0.8100
Hit@1        0.8100
Recall@5     0.8788
MRR@5        0.8773
nDCG@5       0.8574
Hit@5        0.9600
Recall@10    0.9519
MRR@10       0.8800
nDCG@10      0.8808
Hit@10       0.9800
dtype: float64


## 5. 파인튜닝 (train qrels)

각 학습 쿼리와 정답 문서를 쌍으로 만들어 **MultipleNegativesRankingLoss**로 대조학습합니다.
(같은 배치 내 다른 쿼리의 정답을 자동으로 negative 로 사용)

In [11]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

train_examples = []
for qid in train_qids:
    for cid, s in train_gold[qid].items():
        if s > 0 and cid in corpus_text:
            train_examples.append(InputExample(
                texts=[QUERY_PREFIX + queries[qid], PASSAGE_PREFIX + corpus_text[cid]]))
print("학습쌍(query-positive):", len(train_examples))

train_dl = DataLoader(train_examples, shuffle=True, batch_size=FT_BATCH)
train_loss = losses.MultipleNegativesRankingLoss(model)

model.fit(train_objectives=[(train_dl, train_loss)],
          epochs=EPOCHS,
          warmup_steps=max(1, int(0.1 * len(train_dl))),
          show_progress_bar=True)
print("파인튜닝 완료")

학습쌍(query-positive): 680


/tmp/ipykernel_4400/1079733140.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, losses


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


파인튜닝 완료


## 6. (B) 파인튜닝 후 평가

같은 평가셋·같은 문서집합으로 인덱스를 **재구성**(임베딩이 바뀌었으므로)해 평가합니다.

In [12]:
col_after = build_collection("retr-after", model, index_cids)
after = evaluate(col_after, model, eval_qids, gold)
print("파인튜닝 후:"); print(pd.Series(after).round(4))

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

파인튜닝 후:
Recall@1     0.5427
MRR@1        0.8600
nDCG@1       0.8600
Hit@1        0.8600
Recall@5     0.8653
MRR@5        0.8945
nDCG@5       0.8610
Hit@5        0.9500
Recall@10    0.9465
MRR@10       0.8976
nDCG@10      0.8869
Hit@10       0.9700
dtype: float64


## 7. 전/후 비교

In [13]:
comp = pd.DataFrame({"before": before, "after": after})
comp["Δ(after-before)"] = comp["after"] - comp["before"]
comp["개선%"] = np.where(comp["before"] > 0,
                        comp["Δ(after-before)"] / comp["before"] * 100, np.nan)
print("===== 리트리버 파인튜닝 전/후 (동일 평가셋, 축소 코퍼스) =====")
comp.round(4)

===== 리트리버 파인튜닝 전/후 (동일 평가셋, 축소 코퍼스) =====


,before,after,Δ(after-before),개선%
Recall@1,0.5052,0.5427,0.0375,7.4235
MRR@1,0.8100,0.8600,0.0500,6.1728
nDCG@1,0.8100,0.8600,0.0500,6.1728
Hit@1,0.8100,0.8600,0.0500,6.1728
Recall@5,0.8788,0.8653,-0.0136,-1.5443
MRR@5,0.8773,0.8945,0.0172,1.9567
nDCG@5,0.8574,0.8610,0.0035,0.4109
Hit@5,0.9600,0.9500,-0.0100,-1.0417
Recall@10,0.9519,0.9465,-0.0054,-0.5661
MRR@10,0.8800,0.8976,0.0176,1.9995


In [14]:
# 요약: 지표별 개선 여부
improved = (comp["Δ(after-before)"] > 0).sum()
print(f"개선된 지표: {improved} / {len(comp)}")
print("평균 Δ:", round(comp['Δ(after-before)'].mean(), 4))

개선된 지표: 8 / 12
평균 Δ: 0.0161


## 정리 · 다른 도메인으로 옮기기

- **파인튜닝 전/후**를 동일 평가셋·동일 지표(Recall/MRR/nDCG/Hit @k)로 비교했습니다.
- 학습은 `default[train]` 관련도 라벨에서 (query, 정답문서) 쌍을 만들어 MNRL 대조학습.

### 다른 도메인 데이터셋으로 교체할 때
1. 상단 **`DATA` dict** 를 새 데이터셋의 서브셋/컬럼/스플릿 이름으로 수정.
2. 데이터가 (queries + qrels(score>0=정답) + corpus(id,text)) 형식이면 나머지 코드는 그대로 동작.
3. 도메인 언어/특성에 맞게 `BASE_MODEL`(+ 필요 시 `QUERY_PREFIX/PASSAGE_PREFIX`) 만 교체.

### 유의점
- 평가가 **축소 코퍼스**(정답+네거티브 풀) 기준이라 절대수치는 낙관적 → **전/후 상대 비교**가 목적.
- 과적합/성능하락이 보이면 `EPOCHS`, `FT_BATCH`, `N_TRAIN_QUERIES` 조정. train/eval 쿼리는 분리되어 있습니다(train_split vs dev_split).
- 더 강한 세팅: 하드 네거티브 마이닝, e5/bge-m3 등 큰 모델, epoch↑.